# FuseMap Tutorial 2.1: Mapping to new datasets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wanglab-broad/FuseMap/blob/main/docs/notebooks/4_map_new_dataset_customized.ipynb)

**Notebook setup**

**Colab:** choose **Runtime → Change runtime type → T4 GPU** before running the
setup cells. If Colab starts with Python 3.12 or newer, the first cell installs a
Python 3.11 environment and **restarts the notebook session**. Wait for it to
reconnect, then run the first cell again; it should report that Python is ready.
Run the install cell next, then continue with the tutorial. Setup can take several
minutes. A fresh Colab runtime needs setup again.

The setup uses a pinned development revision of
[CondaColab](https://github.com/conda-incubator/condacolab/tree/9df6578d7547f748e22d16b3a5755290bb41b9ad)
to switch the actual notebook kernel. The brief disconnect is expected. If a setup
attempt fails, use **Runtime → Disconnect and delete runtime**, reconnect, and
start again. See the [installation guide](https://fusemap.readthedocs.io/en/latest/install.html#google-colab)
for details and a local-runtime alternative.

**Local Jupyter:** use your existing Python 3.9–3.11 FuseMap environment; these
cells do not replace or automatically install packages into a local environment.


In [ ]:
# Colab: prepare Python 3.11, then reconnect and run this cell again.
import sys
import subprocess

_in_colab = "google.colab" in sys.modules
_supported_python = (3, 9) <= sys.version_info[:2] < (3, 12)
if _in_colab and not _supported_python:
    # Pin the CondaColab revision that supports switching the notebook kernel's Python.
    # The stable 0.1.x release does not provide this python_version API.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
         "https://github.com/conda-incubator/condacolab/archive/"
         "9df6578d7547f748e22d16b3a5755290bb41b9ad.zip"],
        check=True,
    )
    import condacolab
    print("Preparing Python 3.11. After Colab reconnects, run this cell again, then continue.", flush=True)
    condacolab.install(python_version="3.11", dependencies={"numpy": "1.26.*", "matplotlib": "3.8.*"})
elif not _supported_python:
    raise RuntimeError(
        "Local notebooks require Python 3.9-3.11. Select your FuseMap kernel, "
        "or follow https://fusemap.readthedocs.io/en/latest/install.html."
    )
else:
    print(f"Python {sys.version.split()[0]} is ready. Continue to the install cell.")


In [ ]:
import sys
import subprocess

if not (3, 9) <= sys.version_info[:2] < (3, 12):
    raise RuntimeError(
        "Run the Python setup cell above first. Wait for Colab to reconnect, then "
        "rerun that cell and confirm Python 3.11 before installing FuseMap."
    )
if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fusemap[tutorials] @ git+https://github.com/wanglab-broad/FuseMap.git@v1.2.0"],
        check=True,
    )
    import torch
    if torch.cuda.is_available():
        # FuseMap's default DGL wheel is CPU-only. Match torch 2.0.1's CUDA 11.7 build.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "dgl==1.1.1+cu117",
             "--find-links", "https://data.dgl.ai/wheels/cu117/repo.html"],
            check=True,
        )

import fusemap
import torch
import dgl

if "google.colab" in sys.modules and torch.cuda.is_available():
    # Fail here, before downloading data, if the GPU graph backend is unavailable.
    _test_graph = dgl.graph(([0], [1]), num_nodes=2, device="cuda")
    del _test_graph
print(f"Ready: Python {sys.version.split()[0]}, FuseMap {fusemap.__version__}, "
      f"torch {torch.__version__}, DGL {dgl.__version__}; "
      f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


In this tutorial, we will demonstrate how to map new spatial transcriptomics data to an existing FuseMap integration. This is useful when you want to analyze new samples in the context of previously integrated datasets.

FuseMap provides functionality to project new data points into the same latent space as the reference integration, allowing you to:

1. Compare new samples to existing integrated data
2. Transfer annotations and insights from the reference to new data
3. Analyze spatial patterns across old and new datasets together

We use the integrated model trained on merfish and starmap, and then transfer the model to slideseq.

- MERFISH data: Zhang et al. [Nature paper](https://www.nature.com/articles/s41586-023-06808-9#data-availability)
- STARmap data: Shi et al., [Nature paper](https://www.nature.com/articles/s41586-023-06569-5#data-availability)
- Slide-seq data: Langlieb et al., [Nature paper](https://www.nature.com/articles/s41586-023-06818-7#data-availability)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

## 2.1.1 Get the reference model and the new dataset

The reference is the model already trained in **Tutorial 1.1** (MERFISH + STARmap,
`./output_tutorial1`). Keep that tutorial's original `./tutorial1_data` folder too:
we read its expression data once to build reference signatures, without retraining
the reference. The query is a new Slide-seq puck.

For later mapping runs, the saved `reference_signatures.npz` replaces the need to
read those original expression files again.


In [ ]:
import gdown

if not os.path.exists("./output_tutorial1/trained_model"):
    raise FileNotFoundError("Run Tutorial 1.1 first - its ./output_tutorial1 is the reference model here.")
os.makedirs("./tutorial4_data", exist_ok=True)
if not os.path.exists("./tutorial4_data/slideseq_Puck60.h5ad"):
    gdown.download(id="1wCjQSjRYxqf3gHZjtkYA2Xg9-QSoOwtq", output="./tutorial4_data/slideseq_Puck60.h5ad", quiet=False)

## 2.1.2 Map and deconvolve new beads

Declare the Slide-seq query as bead data. FuseMap first adapts the query to the
saved reference, then runs Stage-B with **fixed reference archetypes and expression
signatures**, fitting only the new beads' mixture weights and platform terms.
The reference checkpoint and embeddings remain unchanged.

Use only single-cell sections as `sig_ref`. With a custom reference these declared
sections define both the archetypes and their signatures. Results use one output
subdirectory per input file; canonical cell/tissue embeddings contain the Stage-B
readout. Original mapped embeddings are retained with the `_nodeconv` suffix.


In [ ]:
import fusemap

pretrain_model_path = "./output_tutorial1"
fusemap.map_to_reference(
    "./tutorial4_data", "./output_tutorial4", pretrain_model_path,
    bead_files="slideseq",
    sig_ref="merfish,starmap",
    reference_data_folder_path="./tutorial1_data",
)

output_dir = "./output_tutorial4/slideseq_Puck60.h5ad"


### Inspect the mixture readout

`pi` has one row per bead and one column per retained reference archetype. Rows
sum to one. Archetypes are clusters of reference embeddings, not necessarily
one-to-one cell types. Only genes measured in the reference signatures contribute
to the reconstruction loss. Gene-panel coverage and per-bead reconstruction errors
help assess the limits of this fit.


In [ ]:
import numpy as np

with np.load(os.path.join(output_dir, "stageB_pi.npz"), allow_pickle=False) as mixtures:
    pi = mixtures["pi"]
    bead_ids = mixtures["obs_names"]
    covered = mixtures["gene_mask"]
    reconstruction_mse = mixtures["reconstruction_mse"]

np.testing.assert_allclose(pi.sum(axis=1), 1.0, atol=1e-6)
print(f"{pi.shape[0]} beads, {pi.shape[1]} reference archetypes")
print(f"Reference signatures cover {covered.sum()}/{len(covered)} query genes")
print(f"Median bead reconstruction MSE: {np.median(reconstruction_mse):.4g}")


## 2.1.3 Transfer annotations to the mapped data — one call

The reference embedding (Tutorial 1.1's output) carries STARmap labels; the mapped Slide-seq
puck does not. We concatenate reference + query in the shared latent space and let
`fusemap.transfer_labels` do the rest — for cell types and tissue regions alike.

In [ ]:
import anndata
import numpy as np
import pandas as pd

ref = sc.read_h5ad(os.path.join(pretrain_model_path, "ad_celltype_embedding.h5ad"))
qry = sc.read_h5ad(os.path.join(output_dir, "ad_celltype_embedding.h5ad"))

combo = anndata.AnnData(
    X=np.vstack([np.asarray(ref.X), np.asarray(qry.X)]),
    obs=pd.concat([ref.obs.assign(role="reference"), qry.obs.assign(role="query")]),
)
result = fusemap.transfer_labels(combo, label_key="gt_cell_type_main")
print(f"balanced held-out accuracy (reference): {result['test_accuracy']:.3f}")

qry.obs["cell_type"] = combo.obs["transfer_gt_cell_type_main"].values[ref.n_obs:]
qry.obs["cell_type_uncertainty"] = combo.obs["transfer_gt_cell_type_main_uncertainty"].values[ref.n_obs:]

### Transferred labels in space

Each panel shows a **single classifier label per bead**, assigned from its rebuilt
embedding. These labels summarize similarity to the reference; they are not the
bead's full composition. Use `stageB_pi.npz` for archetype mixture weights.
Spatially coherent patterns support spatial consistency, but do not establish
composition accuracy. The held-out accuracy printed above evaluates reference
cells, not the query beads.


In [ ]:
import matplotlib.pyplot as plt

top_types = qry.obs["cell_type"].value_counts().index[:12]
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
x, y = pd.to_numeric(qry.obs["x"]), pd.to_numeric(qry.obs["y"])
for ax, ct in zip(axes.flat, top_types):
    m = (qry.obs["cell_type"] == ct).values
    ax.scatter(x, y, s=0.5, color="gainsboro")
    ax.scatter(x[m], y[m], s=1.2, color="crimson")
    ax.set_title(str(ct)[:40], fontsize=9); ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
plt.tight_layout(); plt.show()

### Tissue regions, same recipe

In [ ]:
ref_t = sc.read_h5ad(os.path.join(pretrain_model_path, "ad_tissueregion_embedding.h5ad"))
qry_t = sc.read_h5ad(os.path.join(output_dir, "ad_tissueregion_embedding.h5ad"))
combo_t = anndata.AnnData(
    X=np.vstack([np.asarray(ref_t.X), np.asarray(qry_t.X)]),
    obs=pd.concat([ref_t.obs.assign(role="reference"), qry_t.obs.assign(role="query")]),
)
res_t = fusemap.transfer_labels(combo_t, label_key="gt_tissue_region_main")
qry_t.obs["tissue_region"] = combo_t.obs["transfer_gt_tissue_region_main"].values[ref_t.n_obs:]
print(f"balanced held-out accuracy (regions): {res_t['test_accuracy']:.3f}")

fig, ax = plt.subplots(figsize=(8, 8))
xt, yt = pd.to_numeric(qry_t.obs["x"]), pd.to_numeric(qry_t.obs["y"])
for reg in sorted(qry_t.obs["tissue_region"].astype(str).unique()):
    m = (qry_t.obs["tissue_region"].astype(str) == reg).values
    if m.sum() > 50:
        ax.scatter(xt[m], yt[m], s=1.5, label=reg)
ax.set_title("Slide-seq puck: TRANSFERRED tissue regions")
ax.legend(markerscale=8, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
plt.show()

## When to map vs when to integrate

Mapping reuses a fixed reference and trains only query adaptation and, for declared
beads, Stage-B mixture weights/platform terms. Integration jointly trains the input
sections before optional Stage-B. The same mixture objective does not guarantee
identical results: joint training can change the reference embedding and archetypes.

Reference coverage of cell states, anatomy and measured genes limits both mapping
and deconvolution. Compare reconstruction errors, mixture stability and independent
biological evidence; UMAP mixing alone is not an accuracy test. Reference archetype
weights are approximate and are not calibrated cell counts or resolved individual cells.

For another query, reuse the signatures already prepared above:

```python
fusemap.map_to_reference(
    "./next_beads", "./next_mapped", pretrain_model_path,
    bead_files="slideseq",
    reference_signatures_path="./output_tutorial4/reference_signatures.npz",
)
```

Alternatively, prepare the signatures once with
`fusemap.prepare_reference_signatures(pretrain_model_path, "./tutorial1_data",
"merfish,starmap", "./reference_signatures.npz")`.
